# Topic-label embeddings in 2D with t-SNE

Each plot uses the same 1,000 topic paths, color and shape assignments, and t-SNE settings. Each projection is fitted independently, so compare neighborhoods rather than axis positions. Hover over a point for its full topic path.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "topic_embeddings").is_dir())
EMBEDDING_DIR = ROOT / "data/processed/topic_label_embeddings"
SELECTED_TOPICS = ROOT / "data/processed/top_1000_topics.csv"
TOPIC_REFERENCE = ROOT / "results/sgns/topic_embeddings_normalized.csv"
PLOT_DIR = ROOT / "output"
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
PERPLEXITY = 20
RANDOM_STATE = 42

In [13]:
selected = pd.read_csv(SELECTED_TOPICS, usecols=TOPIC_COLUMNS, dtype=str, keep_default_na=False)
if len(selected) != 1000:
    raise ValueError(f"Expected 1,000 selected topic paths, found {len(selected):,}")


marker_symbols = [
    "circle", "square", "diamond", "cross", "x",
    "triangle-up", "triangle-down", "triangle-left", "triangle-right",
    "pentagon", "hexagon", "star", "hexagram", "star-triangle-up",
    "star-triangle-down", "star-square", "star-diamond", "diamond-tall",
    "diamond-wide", "hourglass", "bowtie", "circle-cross", "circle-x",
    "square-cross", "square-x", "diamond-cross", "diamond-x",
]
marker_symbols += [symbol + "-open" for symbol in marker_symbols]
reference = selected
if TOPIC_REFERENCE.exists():
    reference = pd.read_csv(TOPIC_REFERENCE, usecols=TOPIC_COLUMNS[:2], keep_default_na=False)

main_topics = pd.concat([reference["Topic_1"], selected["Topic_1"]]).unique().tolist()
topic2_symbols = {}
for topic1, group in selected.groupby("Topic_1"):
    known = sorted(reference.loc[reference["Topic_1"] == topic1, "Topic_2"].unique())
    labels = known + sorted(set(group["Topic_2"]) - set(known))
    if len(labels) > len(marker_symbols):
        raise ValueError(f"Add more marker symbols for {topic1}")
    topic2_symbols[topic1] = dict(zip(labels, marker_symbols))

In [14]:
def load_variant(variant):
    data_file = EMBEDDING_DIR / variant / "topic_label_embeddings.csv"
    embeddings = pd.read_csv(data_file, keep_default_na=False)
    means = selected.merge(embeddings, on=TOPIC_COLUMNS, how="left", validate="one_to_one", indicator=True)
    missing = means.loc[means["_merge"] == "left_only", TOPIC_COLUMNS]
    if not missing.empty:
        raise ValueError(f"Missing label embeddings for {len(missing)} selected paths:\n{missing.head()}")
    means = means.drop(columns="_merge")
    means["Topic_depth"] = means[TOPIC_COLUMNS].ne("").sum(axis=1)
    dimensions = [c for c in means if c.startswith("dim_")]
    vectors = normalize(means[dimensions].to_numpy(dtype=np.float32))
    projection_df = means[[*TOPIC_COLUMNS, "Topic_depth"]].copy()
    projection_df["Topic_path"] = means[TOPIC_COLUMNS].apply(
        lambda row: " / ".join(label for label in row if label), axis=1
    )
    return projection_df, vectors


def plot_variant(variant, title):
    projection_df, vectors = load_variant(variant)
    tsne = TSNE(
        n_components=2, perplexity=PERPLEXITY, init="pca", learning_rate="auto",
        metric="euclidean", max_iter=1_000, random_state=RANDOM_STATE,
    )
    if len(vectors) <= PERPLEXITY:
        raise ValueError("PERPLEXITY must be smaller than the number of topics")
    projection_df[["TSNE_1", "TSNE_2"]] = tsne.fit_transform(vectors)
    print(f"Projected {len(vectors):,} topic paths from {vectors.shape[1]} dimensions.")
    fig = px.scatter(
        projection_df,
        x="TSNE_1", y="TSNE_2", color="Topic_1", symbol="Topic_2",
        category_orders={"Topic_1": main_topics},
        custom_data=["Topic_1", "Topic_2"],
        render_mode="svg",
        hover_name="Topic_path",
        hover_data={"Topic_depth": ":,", "Topic_1": False, "Topic_2": False, "TSNE_1": False, "TSNE_2": False},
        title=f"Top 1,000 topic-label embeddings · {title} · t-SNE (perplexity={PERPLEXITY}, seed={RANDOM_STATE})",
        labels={"Topic_1": "Main topic", "Topic_depth": "Topic depth", "Topic_2": "Subtopic", "TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"},
        template="plotly_white",
        height=800,
    )
    # Each trace contains one (Topic_1, Topic_2) pair.
    # Show one legend entry per main topic; clicking it toggles all its subtopics.
    seen_topic1 = set()
    for trace in fig.data:
        topic1, topic2 = trace.customdata[0][:2]
        trace.marker.symbol = topic2_symbols[topic1][topic2]
        trace.name = topic1
        trace.legendgroup = topic1
        trace.showlegend = topic1 not in seen_topic1
        seen_topic1.add(topic1)
    fig.update_traces(marker=dict(size=7, opacity=0.8))
    fig.update_layout(legend_title_text="Main topic", legend_groupclick="togglegroup")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(PLOT_DIR / f"topic_label_embeddings_{variant}_tsne.html", include_plotlyjs=True)
    fig.show()

## Full topic path

In [15]:
plot_variant("full_path", "Full path")

Projected 1,000 topic paths from 5120 dimensions.


## Deepest topic label

In [16]:
plot_variant("leave_only", "Deepest label")

Projected 1,000 topic paths from 5120 dimensions.


## Weighted topic labels

In [17]:
weighted_file = EMBEDDING_DIR / "weighted" / "topic_label_embeddings.csv"
if weighted_file.is_file():
    plot_variant("weighted", "Weighted labels")
else:
    print(f"Weighted embeddings pending. Save the CSV to {weighted_file} and rerun this cell.")

Projected 1,000 topic paths from 5120 dimensions.


## Nearest neighbors in the three original embedding spaces

In [18]:
query_topic = "wikileaks"
rank = pd.RangeIndex(1, 11, name="Rank")
rankings = {}
for variant, title in [("full_path", "Full path"), ("leave_only", "Deepest label"), ("weighted", "Weighted")]:
    if variant == "weighted" and not (EMBEDDING_DIR / variant / "topic_label_embeddings.csv").is_file():
        rankings[title] = pd.DataFrame({"Status": ["CSV pending"] + [""] * 9}, index=rank)
        continue

    topics, vectors = load_variant(variant)
    query = topics["Topic_2"].eq(query_topic).to_numpy()
    if not query.any():
        raise ValueError(f"No selected topic has Topic_2={query_topic!r}")
    centroid = normalize(vectors[query].mean(axis=0, keepdims=True))[0]
    candidates = np.flatnonzero(~query)
    scores = vectors[candidates] @ centroid
    order = np.argsort(-scores, kind="stable")[:len(rank)]
    rankings[title] = pd.DataFrame({
        "Topic path": topics["Topic_path"].iloc[candidates[order]].to_numpy(),
        "Cosine": scores[order],
    }, index=rank[:len(order)])

neighbors = pd.concat(rankings, axis=1)
neighbors

Full path            \
                              Topic path    Cosine   
Rank                                                 
1      web / netzpolitik / wikileaks2010  0.677776   
2             international / diplomatie  0.667962   
3          international / abhoerskandal  0.639854   
4                    international / uno  0.596964   
5     international / flucht-und-politik  0.589978   
6               international / ruestung  0.586646   
7             international / jihadismus  0.586620   
8                   international / nato  0.554579   
9         international / europaszukunft  0.549293   
10                   international / g20  0.542845   

                                     Deepest label            \
                                        Topic path    Cosine   
Rank                                                           
1                web / netzpolitik / wikileaks2010  0.798878   
2                    international / abhoerskandal  0.456748   
3                               inland / watchblog  0.416216   
4                          web / webmix / webwatch  0.400386   
5           inland / bundeslaender / wienerpolitik  0.383384   
6                     etat / medien / charliehebdo  0.377816   
7                           inland / coronapolitik  0.377636   
8     panorama / weltchronik / korrespondentenblog  0.374172   
9                     web / innovationen / webtalk  0.373830   
10         international / europa / anschlagberlin  0.363939   

                                Weighted            
                              Topic path    Cosine  
Rank                                                
1          international / abhoerskandal  0.784736  
2             international / diplomatie  0.752257  
3             international / jihadismus  0.725941  
4     international / flucht-und-politik  0.724807  
5                    international / g20  0.709498  
6                   international / nato  0.706689  
7         international / europaszukunft  0.701499  
8     international / friedensnobelpreis  0.686803  
9            international / ub-eastblog  0.677744  
10     web / netzpolitik / wikileaks2010  0.676050